In [ ]:
# bootstrap: Colab (private clone via GH_TOKEN) + local import of `agentcore` (auto-inserted)
import sys, pathlib
if "google.colab" in sys.modules:
    import os, subprocess
    _slug = "aniryou/full-stack-agentic-engineer"
    _repo = pathlib.Path("/content/full-stack-agentic-engineer")
    if not _repo.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret; add a GitHub token (repo scope) as Colab secret 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_repo)], check=True)
        subprocess.run(["git", "-C", str(_repo), "remote", "set-url", "origin", f"https://github.com/{_slug}.git"])
    os.chdir(_repo / "07-application-agent-framework/agent-fundamentals/agent-core")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
_r = pathlib.Path.cwd().resolve()
while _r != _r.parent and not (_r / "agentcore").exists():
    _r = _r.parent
if str(_r) not in sys.path:
    sys.path.insert(0, str(_r))
del _r

# 04 · A mini support agent (capstone)

Put the three ideas together into a small but honest support agent for a bank:

* it answers **only** from tool results (never invents a balance);
* reads are free; a **card block** is irreversible, so it needs human approval;
* anything it can't do becomes a **case** for a human.

This is the same shape as the primer's §8.1 bank scenario, shrunk to what fits on one
screen. The production version (identity, MCP, evals, tracing) is the separate
`gcp-agent-platform-lab` repo — this is the concept underneath it.

In [ ]:
from agentcore import Agent, FakeLLM, ToolError, call, calls, text, tool

# -- the systems of record (fakes) --------------------------------------------
ACCOUNTS = {"a1": {"balance": 1234.5, "currency": "SGD"}}
CARDS = {"card-1": {"status": "active", "account": "a1"}}
CASES = []

@tool
def get_balance(account_id: str) -> dict:
    """Return the balance for an account."""
    if account_id not in ACCOUNTS:
        raise ToolError("no such account", kind="not_found", hint="Confirm the account id.")
    return {"account_id": account_id, **ACCOUNTS[account_id]}

@tool(confirm=True)                       # irreversible → the loop will ask a human
def block_card(card_id: str) -> dict:
    """Block a card. Irreversible; requires approval."""
    CARDS[card_id]["status"] = "blocked"
    return {"card_id": card_id, "status": "blocked"}

## Exercise 4.1 — the escalation tool

Write `raise_case(summary: str, queue: str)` that appends `{"summary","queue"}` to the
`CASES` list and returns `{"case_id": ..., "queue": ...}` where `case_id` is
`"CASE-" + <new length of CASES>`. This is the "hand to a human" path for anything out
of scope.

In [ ]:
@tool
def raise_case(summary: str, queue: str) -> dict:
    """Open a case for a human team."""
    CASES.append({"summary": summary, "queue": queue})
    return {"case_id": f"CASE-{len(CASES)}", "queue": queue}

In [ ]:
CASES.clear()
out = raise_case.run({"summary": "dispute on ORD-1", "queue": "billing"})
assert out["data"]["case_id"] == "CASE-1" and len(CASES) == 1
print("✅ raise_case works:", out["data"])

## The agent
A scripted model walks three intents. Read the transcript for each.

In [ ]:
support = Agent(
    tools=[get_balance, block_card, raise_case],
    instruction="You are a bank support agent. Use tools for every fact; never guess.",
    llm=FakeLLM([]),   # replaced per-run below
)

def run_once(script, message, on_confirm=None):
    support.llm = FakeLLM(script)
    r = support.run(message, on_confirm=on_confirm)
    print(f"\n=== {message} ===")
    print(r.transcript())
    return r

# intent 1: a balance question, answered from the tool
run_once([call("get_balance", account_id="a1"), "Your balance is SGD 1,234.50."],
         "what's my balance on a1?")

## Exercise 4.2 — the approval gate

Run a "block my card" conversation twice with the same script
`[call("block_card", card_id="card-1"), "Your card is now blocked."]`:

* once with `on_confirm` returning **False** — assert the card is still `"active"`
  and the tool result the model saw contains `"declined"`;
* once with `on_confirm` returning **True** — assert the card becomes `"blocked"`.

In [ ]:
script = [call("block_card", card_id="card-1"), "Your card is now blocked."]
CARDS["card-1"]["status"] = "active"
declined = run_once(list(script), "block card-1", on_confirm=lambda name, args: False)
assert CARDS["card-1"]["status"] == "active"
assert any("declined" in m["content"] for m in declined.messages if m["role"] == "tool")

approved = run_once(list(script), "block card-1", on_confirm=lambda name, args: True)
assert CARDS["card-1"]["status"] == "blocked"

In [ ]:
assert CARDS["card-1"]["status"] == "blocked", "after approval the card must be blocked"
print("✅ irreversible action gated on human approval")

## Exercise 4.3 — never act without a tool

The one property that makes this agent trustworthy: every factual claim comes from a
tool result, and every state change comes from a tool call. Write `used_a_tool(result)`
that returns True if the transcript contains at least one `tool` message. Then confirm
the balance answer used one. (In real evals this is a *trajectory* check — did the
agent look before it spoke — and it is graded as an absolute gate.)

In [ ]:
def used_a_tool(result) -> bool:
    return any(m["role"] == "tool" for m in result.messages)

In [ ]:
r = run_once([call("get_balance", account_id="a1"), "SGD 1,234.50."], "balance on a1?")
assert used_a_tool(r) is True
# and an out-of-scope ask should escalate, not invent an answer
esc = run_once([call("raise_case", summary="wants a loan", queue="lending"),
                "I've raised this with our lending team."], "can I get a loan?")
assert used_a_tool(esc) and any(m["name"] == "raise_case" for m in esc.messages if m["role"] == "tool")
print("✅ the agent acts only through tools")

## Where to go next
You now have the whole core: a model, tools, the loop, state, budgets, and a human in
the loop for irreversible actions. The step-up — async and parallel tool calls, MCP
servers and a policy gateway, OAuth identity propagation, evaluation gates, and
tracing — is the `gcp-agent-platform-lab` repository. Same concepts; more machinery.

## The one-minute version
Walk this design as: unit of work (a support interaction), every fact from a tool,
irreversible actions gated by a human, out-of-scope work escalated as a case, and the
trustworthiness property ("never acts without a tool") checked as an absolute eval gate.